In [ ]:
import ollama
import pandas as pd
from tqdm import tqdm
from datasets import load_dataset
import numpy as np

# Giữ nguyên các hàm template của bạn
def apply_qwen_math_template(question: str):
    return (
        "<|im_start|>system\nPlease reason step by step, and put your final answer within \\boxed{}.<|im_end|>\n<|im_start|>user\n"
        + question
        + "<|im_end|>\n<|im_start|>assistant\n"
    )

def apply_r1_template(question: str):
    return (
        "A conversation between User and Assistant. The User asks a question, and the Assistant solves it. The Assistant first thinks about the reasoning process in the mind and then provides the User with the answer. "
        "The reasoning process is enclosed within <think> </think> and answer is enclosed within <answer> </answer> tags, respectively, i.e., <think> reasoning process here </think> <answer> answer here </answer>.\nUser: "
        + question
        + "\nAssistant: <think>"
    )

# Viết lại hàm inference dùng Ollama
def inference_500_questions_ollama(template_type, model_name, dataset):
    all_problems = dataset["problem"].tolist()

    # Áp dụng template
    if template_type == "qwen_math_template":
        prompts = [apply_qwen_math_template(p) for p in all_problems]
    elif template_type == "r1_template":
        prompts = [apply_r1_template(p) for p in all_problems]
    else:
        prompts = all_problems

    print(f"Bắt đầu xử lý {len(prompts)} mẫu với pass@8 bằng model {model_name}...")

    generated_results = []

    # Duyệt qua từng câu hỏi
    for prompt in tqdm(prompts, desc="Đang Inference"):
        answers = []
        # Lặp 8 lần để lấy 8 câu trả lời (tương đương n=8)
        for _ in range(8):
            response = ollama.generate(
                model=model_name,
                prompt=prompt,
                options={
                    "temperature": 0.6,
                    "top_p": 0.95,
                    "num_predict": 2048, # Tương đương max_tokens
                }
            )
            answers.append(response['response'])
        
        generated_results.append(answers)

    # Tạo DataFrame lưu vào CSV
    data_rows = []
    for q, answers in zip(dataset["problem"], generated_results):
        row = {"Question": q}
        for i in range(8):
            row[f"Answer_{i+1}"] = answers[i]
        data_rows.append(row)

    df = pd.DataFrame(data_rows)

    filename = f"ket_qua_{template_type}_pass8.csv"
    df.to_csv(filename, index=False, encoding='utf-8-sig')

    print(f"Xong! Lưu tại {filename}")
    return df

# --- CHẠY THỬ ---
# Lưu ý: "llama3.1" là tên model bạn đã pull bằng terminal lệnh `ollama run llama3.1`
# Thay bằng "qwen2.5-math" hoặc tên khác nếu bạn dùng model khác.
df_results = inference_500_questions_ollama("r1_template", "llama3.1", sample_dataset)